# Exploring S3D8 on artificial data

In [53]:
import json
from pathlib import Path

import torch
from torch import Tensor
import tqdm

## Example

In [54]:
def kmeansplusplus_init(tensor: Tensor, n: int, *, progress: bool) -> Tensor:
    midpoints = [tensor[torch.randint(0, tensor.shape[0], (1,)).item()]]
    for _ in tqdm.tqdm(list(range(1, n)), disable=not progress, desc="k-means++ init"):
        midpoint_tensor = torch.stack(midpoints)
        # idx = torch.cdist(tensor, midpoint_tensor).argmin(-1)
        # probabilities = (tensor - midpoint_tensor[idx]).square().sum(dim=1)
        probabilities = torch.cdist(tensor, midpoint_tensor).square().amin(dim=1)
        probabilities /= probabilities.sum()
        next_midpoint = tensor[torch.multinomial(probabilities, 1).item()]
        midpoints.append(next_midpoint)
    return torch.stack(midpoints)


def k_means(t: Tensor, n_centroids: int, threshold: float, progress: bool = True) -> Tensor:
    _, dim = t.shape
    t = t.float()
    # centroids = t[torch.randperm(t.shape[0], device=t.device)[:n_centroids]].clone()
    centroids = kmeansplusplus_init(t, n_centroids, progress=progress)
    idx = torch.zeros(t.shape[0], device=t.device, dtype=torch.int64)
    last_idx = torch.full_like(idx, -1)
    with tqdm.tqdm(desc="k-means", disable=not progress) as pbar:
        while (last_idx != idx).float().mean().item() > threshold:
            last_idx, idx = idx, torch.cdist(t, centroids).argmin(-1)
            centroids.scatter_reduce_(
                0, idx[:, None].expand(-1, dim), t, "mean", include_self=False
            )
            pbar.update(1)
    return centroids


# Generate data
torch.manual_seed(0)
dist = torch.distributions.Normal(0, 1)
# dist = torch.distributions.StudentT(df=5)
data = dist.sample((2**16, 3))

# Format construction
centroids = k_means(data.abs(), 32, 1e-4)

# Quantisation
q_abs_idx = torch.cdist(data.abs(), centroids).argmin(-1)
q_sign = (data < 0)

reconstructed = centroids[q_abs_idx] * torch.where(q_sign, -1, 1)

# Encoding
code_data = (
    (q_abs_idx << 1)
    | q_sign[:, 0] | (q_sign[:, 1] << 6) | ((q_sign[:, 0] ^ q_sign[:, 2]) << 7)
).byte()
code_tables = torch.stack([
    torch.stack([centroids[:, 0], -centroids[:, 0]], dim=1).view(-1),  # sign-minor
    torch.stack([centroids[:, 1], -centroids[:, 1]], dim=0).view(-1),  # sign-major
    torch.stack([centroids[:, 2], -centroids[:, 2]], dim=1).view(-1),  # sign-minor
])

# Decoding
idx0 = code_data & 0x3f
idx1 = (code_data >> 1) & 0x3f
idx2 = idx0 ^ (code_data >> 7)
decoded = torch.stack([
    code_tables[0][idx0.long()],
    code_tables[1][idx1.long()],
    code_tables[2][idx2.long()],
], dim=1)

# Checks
assert torch.equal(decoded, reconstructed)
print("RMSE:", (decoded - data).pow(2).mean().sqrt().item())

k-means++ init: 100%|██████████| 31/31 [00:00<00:00, 322.14it/s]
k-means: 141it [00:01, 119.43it/s]

RMSE: 0.2092100828886032


## `centroid_data.json`

In [ ]:
seed = 100
distributions = [
    torch.distributions.Normal(0, 1),
    torch.distributions.StudentT(df=7),
]
methods = [
    "1D(8/3)",
    "3D8",
    "S3D8",
]
rows = []
for dist in distributions:
    print(f"# {dist}")
    for method in methods:
        torch.manual_seed(seed)
        data = dist.sample((2**18, 3))
        data /= data.square().mean().sqrt()  # normalise to unit variance
        if method == "1D(8/3)":
            centroids = k_means(data.view(-1, 1), 6, 1e-4)
            q = centroids[torch.cdist(data.view(-1, 1), centroids).argmin(-1)].view_as(data)
            dimension, signed = 1, False
        if method == "3D8":
            centroids = k_means(data, 256, 1e-4)
            q = centroids[torch.cdist(data, centroids).argmin(-1)]
            dimension, signed = 3, False
        if method == "S3D8":
            centroids = k_means(data.abs(), 32, 1e-4)
            # note: copysign doesn't introduce zeros from the sign operand
            q = torch.copysign(centroids[torch.cdist(data.abs(), centroids).argmin(-1)], data)
            dimension, signed = 3, True
        rmse = (q - data).pow(2).mean().sqrt().item()
        print(f"{method}: {rmse}")
        args = {k: v.item() for k, v in dist.__dict__.items() if not k.startswith("_")}
        rows.append(dict(
            distribution=dist.__class__.__name__,
            method=method,
            dimension=dimension,
            signed=signed,
            centroids=centroids.tolist(),
            rmse=rmse,
            **args,
        ))

out = Path("data")
out.mkdir(exist_ok=True)
(out / f"centroid_data.json").write_text(json.dumps(rows));

# Normal(loc: 0.0, scale: 1.0)


k-means++ init: 100%|██████████| 5/5 [00:00<00:00, 48.88it/s]
k-means: 101it [00:01, 64.70it/s]


1D(8/3): 0.2408204823732376


k-means++ init: 100%|██████████| 255/255 [00:15<00:00, 16.39it/s]
k-means: 363it [00:39,  9.09it/s]


3D8: 0.20423972606658936


...

k-means++ init: 100%|██████████| 255/255 [00:16<00:00, 15.27it/s]
k-means: 245it [00:31,  7.70it/s]


3D8: 0.2170853316783905


k-means++ init: 100%|██████████| 31/31 [00:00<00:00, 35.07it/s]
k-means: 200it [00:07, 28.21it/s]

S3D8: 0.224739208817482


37766